[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/089.%20The%20Single%20vs.%20Dual%20Sourcing%20Problem/P89-Tier-3_executed.ipynb)

---


# 89. The Single vs. Dual Sourcing Problem

## Tier 3: Ant Colony Optimization

### Purpose and scope

This notebook uses Ant Colony Optimization (ACO), a population-based metaheuristic, to construct sourcing plans and improve them through repeated pheromone learning. Its inputs are component demand and criticality, supplier costs, reliability, quality scores, and capacity-utilization indicators. Its outputs include a best-found allocation, its cost and weighted scores, convergence data, and visual summaries.

The example is a stochastic search, not a proof of global optimality. Its result depends on random choices, and this notebook does not set a random seed. Re-running it can therefore produce a different search history or best-found solution.

### Modeling assumptions

- Every component is assigned either to one supplier or to two distinct suppliers.
- A dual-sourced component uses a fixed 70%/30% allocation. The construction rule gives components with criticality above 0.7 a 30% chance of being dual-sourced; other components are single-sourced.
- Supplier choice is guided by pheromone and a heuristic based on cost, reliability, quality score, and unused capacity indicator.
- The fitness objective combines normalized cost, reliability, and quality as specified below.
- Annual capacity limits, minimum quality requirements, lead times, and disruption scenarios are data fields but are not enforced as constraints or separately optimized. Capacity utilization affects the heuristic only. Quality scores affect both the heuristic and fitness; `quality_requirement` is not a hard feasibility condition.

These assumptions define the particular ACO construction and evaluation rules used here. They should be considered when interpreting a reported sourcing plan.

### How the search works

An artificial ant constructs one complete sourcing plan by making a supplier choice for each component. Pheromone records how strongly the search has favored component–supplier assignments; heuristic information represents their immediate desirability. Ants use both signals to choose suppliers. After all ants have constructed plans in an iteration, pheromone evaporates and each plan deposits pheromone in proportion to its allocation and fitness. Evaporation limits the persistence of old information, while repeated deposits reinforce assignments appearing in better plans.

Let $C$ be the set of components and $S$ the set of suppliers. For component $c$ and supplier $s$, let $\tau_{cs}$ be pheromone and $\eta_{cs}$ be heuristic information. The desirability used by the choice rule is

$$
D_{cs} = \tau_{cs}^{\alpha}\eta_{cs}^{\beta}.
$$

With probability $q_0$, the ant selects an available supplier with the largest $D_{cs}$ (exploitation). Otherwise, it selects probabilistically using pairwise transition scores derived from the same desirabilities (exploration). In this implementation, the exploration score for one supplier is the mean of its pairwise transition probabilities against the other available suppliers, followed by normalization across available suppliers.

For an allocation $x_{cs}$, where $x_{cs}$ is the share of component $c$ assigned to supplier $s$, the construction rule ensures $\sum_{s\in S}x_{cs}=1$. The implemented scores are

$$
\begin{aligned}
\text{Cost} &= \sum_{c\in C}\sum_{s\in S} d_c x_{cs} k_{cs},\\
\bar r &= \frac{\sum_{c\in C}\sum_{s\in S}d_c x_{cs}r_s}{\sum_{c\in C}d_c},\\
\bar q &= \frac{\sum_{c\in C}\sum_{s\in S}d_c x_{cs}q_{cs}}{\sum_{c\in C}d_c},\\
F &= w_k\frac{\text{Cost}}{10000} + w_r(1-\bar r) + 0.1(1-\bar q).
\end{aligned}
$$

Here, $d_c$ is annual demand, $k_{cs}$ is unit cost, $r_s$ is supplier reliability, and $q_{cs}$ is the supplier's quality score for the component. The weights $w_k$ and $w_r$ are the supplied cost and reliability weights normalized to sum to one. Lower fitness $F$ is preferred. The quality term has a fixed coefficient of 0.1; it is not part of that weight normalization. The cost divisor of 10000 is a scale factor used by this example, not a currency conversion.

For each iteration, pheromone is first reduced by the evaporation rate $\rho$ and then receives deposits. A solution with fitness $F$ deposits $1/(1+F)$, multiplied by each assigned allocation share. Thus, for a component–supplier pair, the deposit is accumulated over the ants that use that pair. The implementation records the best solution found across all iterations; it does not certify that this solution is globally optimal.

If there are $I$ iterations, $A$ ants, $|C|$ components, and $|S|$ suppliers, constructing plans and evaluating their fitness takes approximately $O(IA|C||S|)$ work. The exploration rule compares available suppliers pairwise, so its supplier-choice calculations can add up to $O(IA|C||S|^2)$ work in the worst case. The stored pheromone table has $O(|C||S|)$ entries.

### What to examine

- The best-so-far fitness and the range of fitness values across ants.
- Whether pheromone becomes concentrated on a subset of assignments.
- The cost, reliability, and quality trade-offs in the best-found plan and simple single-sourcing reference plans.
- The number of components assigned to one or two suppliers. This is not the same as measuring diversity among all possible allocation plans.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from collections import defaultdict
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class ACOComponent:
    """Component data used to construct and evaluate sourcing plans."""

    name: str
    criticality_score: float
    annual_demand: int
    quality_requirement: float


@dataclass
class ACOSupplier:
    """Supplier data used by the ACO construction and evaluation rules."""

    name: str
    unit_costs: Dict[str, float]
    reliability_score: float
    quality_scores: Dict[str, float]
    capacity_utilization: float
    lead_time: int
    max_capacity: int


@dataclass
class ACOSolution:
    """One sourcing plan and its evaluated objective components."""

    allocations: Dict[str, Dict[str, float]]
    fitness: float
    total_cost: float
    reliability_score: float
    quality_score: float
    iteration: int
    ant_id: int

In [ ]:
class AntColonyOptimizer:
    """Ant Colony Optimization for the stated single/dual sourcing rules."""

    def __init__(
        self,
        components: List[ACOComponent],
        suppliers: List[ACOSupplier],
        n_ants: int = 30,
        n_iterations: int = 100,
        alpha: float = 1.0,
        beta: float = 2.0,
        rho: float = 0.1,
        q0: float = 0.9,
        cost_weight: float = 0.6,
        reliability_weight: float = 0.4,
    ):
        """Initialize the colony, problem data, and pheromone table.

        Args:
            components: Components whose demand must be allocated.
            suppliers: Suppliers available for each component.
            n_ants: Number of plans constructed per iteration.
            n_iterations: Number of pheromone-update iterations.
            alpha: Importance assigned to pheromone in supplier desirability.
            beta: Importance assigned to heuristic information.
            rho: Pheromone evaporation rate.
            q0: Probability of selecting the highest-desirability supplier.
            cost_weight: Cost weight before normalization with reliability_weight.
            reliability_weight: Reliability weight before normalization.
        """
        if not components:
            raise ValueError("At least one component is required.")
        if not suppliers:
            raise ValueError("At least one supplier is required.")
        if n_ants <= 0 or n_iterations <= 0:
            raise ValueError("n_ants and n_iterations must be positive.")
        if not 0 <= rho <= 1:
            raise ValueError("rho must be between 0 and 1.")
        if not 0 <= q0 <= 1:
            raise ValueError("q0 must be between 0 and 1.")
        if alpha < 0 or beta < 0:
            raise ValueError("alpha and beta must be non-negative.")
        if cost_weight < 0 or reliability_weight < 0:
            raise ValueError("Objective weights must be non-negative.")
        total_weight = cost_weight + reliability_weight
        if total_weight <= 0:
            raise ValueError("At least one objective weight must be positive.")

        component_names = [component.name for component in components]
        supplier_names = [supplier.name for supplier in suppliers]
        if len(set(component_names)) != len(component_names):
            raise ValueError("Component names must be unique.")
        if len(set(supplier_names)) != len(supplier_names):
            raise ValueError("Supplier names must be unique.")

        for component in components:
            if component.annual_demand < 0:
                raise ValueError("Annual demand must be non-negative.")
            for supplier in suppliers:
                if component.name not in supplier.unit_costs:
                    raise ValueError(f"Missing unit cost for {component.name}.")
                if component.name not in supplier.quality_scores:
                    raise ValueError(f"Missing quality score for {component.name}.")

        if sum(component.annual_demand for component in components) <= 0:
            raise ValueError("Total annual demand must be positive.")

        self.components = components
        self.suppliers = suppliers
        self.n_ants = n_ants
        self.n_iterations = n_iterations
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.q0 = q0
        self.cost_weight = cost_weight / total_weight
        self.reliability_weight = reliability_weight / total_weight

        self.pheromone = defaultdict(lambda: defaultdict(float))
        self.initialize_pheromone()
        self.best_solution: Optional[ACOSolution] = None
        self.best_fitness = float("inf")
        self.iteration_best_solutions: List[ACOSolution] = []
        self.convergence_history: List[Dict[str, float]] = []

    def initialize_pheromone(self, initial_pheromone: float = 0.1) -> None:
        """Give every component–supplier assignment the same starting trail."""
        if initial_pheromone <= 0:
            raise ValueError("initial_pheromone must be positive.")
        for component in self.components:
            for supplier in self.suppliers:
                self.pheromone[component.name][supplier.name] = initial_pheromone

    def calculate_heuristic_info(
        self, component: ACOComponent, supplier: ACOSupplier
    ) -> float:
        """Combine cost, reliability, quality, and unused-capacity indicators."""
        unit_cost = supplier.unit_costs[component.name]
        cost_desirability = 1.0 / (1.0 + unit_cost / 10.0)
        reliability_desirability = supplier.reliability_score
        quality_desirability = supplier.quality_scores[component.name]
        capacity_desirability = 1.0 - supplier.capacity_utilization
        return (
            cost_desirability
            + reliability_desirability
            + quality_desirability
            + capacity_desirability
        ) / 4.0

    def calculate_transition_probability(
        self,
        component: ACOComponent,
        supplier1: ACOSupplier,
        supplier2: ACOSupplier,
    ) -> float:
        """Compare two suppliers using their pheromone-weighted desirabilities."""
        pheromone1 = self.pheromone[component.name][supplier1.name] ** self.alpha
        heuristic1 = self.calculate_heuristic_info(component, supplier1) ** self.beta
        pheromone2 = self.pheromone[component.name][supplier2.name] ** self.alpha
        heuristic2 = self.calculate_heuristic_info(component, supplier2) ** self.beta

        numerator = pheromone1 * heuristic1
        denominator = numerator + pheromone2 * heuristic2
        return numerator / denominator if denominator > 0 else 0.5

    def select_supplier_for_component(
        self,
        component: ACOComponent,
        available_suppliers: List[ACOSupplier],
    ) -> Tuple[ACOSupplier, bool]:
        """Select a supplier and indicate whether it was the best-known choice."""
        if not available_suppliers:
            raise ValueError("At least one supplier must be available.")
        if len(available_suppliers) == 1:
            return available_suppliers[0], True

        def desirability(supplier: ACOSupplier) -> float:
            pheromone = self.pheromone[component.name][supplier.name] ** self.alpha
            heuristic = self.calculate_heuristic_info(component, supplier) ** self.beta
            return pheromone * heuristic

        best_supplier = max(available_suppliers, key=desirability)

        if random.random() < self.q0:
            return best_supplier, True

        probabilities = []
        for supplier in available_suppliers:
            other_suppliers = [
                other for other in available_suppliers if other != supplier
            ]
            pairwise_scores = [
                self.calculate_transition_probability(component, supplier, other)
                for other in other_suppliers
            ]
            probabilities.append(sum(pairwise_scores) / len(pairwise_scores))

        total_probability = sum(probabilities)
        if total_probability > 0:
            probabilities = [value / total_probability for value in probabilities]
        else:
            probabilities = [1.0 / len(available_suppliers)] * len(available_suppliers)

        selected_index = int(np.random.choice(len(available_suppliers), p=probabilities))
        selected_supplier = available_suppliers[selected_index]
        return selected_supplier, selected_supplier == best_supplier

    def construct_solution(self, ant_id: int, iteration: int) -> ACOSolution:
        """Construct and evaluate one complete sourcing plan for an ant."""
        allocations: Dict[str, Dict[str, float]] = {}

        for component in self.components:
            use_dual_sourcing = (
                component.criticality_score > 0.7 and random.random() < 0.3
            )

            if use_dual_sourcing and len(self.suppliers) >= 2:
                available_suppliers = self.suppliers.copy()
                random.shuffle(available_suppliers)
                primary_supplier, _ = self.select_supplier_for_component(
                    component, available_suppliers
                )
                available_suppliers.remove(primary_supplier)
                secondary_supplier, _ = self.select_supplier_for_component(
                    component, available_suppliers
                )
                allocations[component.name] = {
                    primary_supplier.name: 0.7,
                    secondary_supplier.name: 0.3,
                }
            else:
                selected_supplier, _ = self.select_supplier_for_component(
                    component, self.suppliers
                )
                allocations[component.name] = {selected_supplier.name: 1.0}

        fitness, total_cost, reliability_score, quality_score = (
            self.calculate_solution_fitness(allocations)
        )
        return ACOSolution(
            allocations=allocations,
            fitness=fitness,
            total_cost=total_cost,
            reliability_score=reliability_score,
            quality_score=quality_score,
            iteration=iteration,
            ant_id=ant_id,
        )

    def calculate_solution_fitness(
        self, allocations: Dict[str, Dict[str, float]]
    ) -> Tuple[float, float, float, float]:
        """Return fitness, total cost, demand-weighted reliability, and quality."""
        total_cost = 0.0
        weighted_reliability = 0.0
        weighted_quality = 0.0
        total_demand = sum(component.annual_demand for component in self.components)

        for component in self.components:
            component_allocations = allocations[component.name]
            if not component_allocations:
                raise ValueError(f"No supplier allocation for {component.name}.")
            if any(value < 0 for value in component_allocations.values()):
                raise ValueError("Allocation shares must be non-negative.")
            if not np.isclose(sum(component_allocations.values()), 1.0):
                raise ValueError(
                    f"Allocation shares for {component.name} must sum to 1."
                )

            for supplier_name, allocation in component_allocations.items():
                supplier = next(
                    (item for item in self.suppliers if item.name == supplier_name),
                    None,
                )
                if supplier is None:
                    raise ValueError(f"Unknown supplier: {supplier_name}.")

                demand_share = component.annual_demand * allocation
                total_cost += demand_share * supplier.unit_costs[component.name]
                weighted_reliability += demand_share * supplier.reliability_score
                weighted_quality += (
                    demand_share * supplier.quality_scores[component.name]
                )

        average_reliability = weighted_reliability / total_demand
        average_quality = weighted_quality / total_demand
        normalized_cost = total_cost / 10000.0
        normalized_reliability = 1.0 - average_reliability
        normalized_quality = 1.0 - average_quality
        fitness = (
            self.cost_weight * normalized_cost
            + self.reliability_weight * normalized_reliability
            + 0.1 * normalized_quality
        )
        return fitness, total_cost, average_reliability, average_quality

    def update_pheromones(self, solutions: List[ACOSolution]) -> None:
        """Evaporate existing trails, then reinforce assignments in each plan."""
        for component in self.components:
            for supplier in self.suppliers:
                self.pheromone[component.name][supplier.name] *= 1.0 - self.rho

        for solution in solutions:
            pheromone_deposit = 1.0 / (1.0 + solution.fitness)
            for component_name, component_allocations in solution.allocations.items():
                for supplier_name, allocation in component_allocations.items():
                    self.pheromone[component_name][supplier_name] += (
                        pheromone_deposit * allocation
                    )

    def optimize(self) -> ACOSolution:
        """Run the ant construction and pheromone-update process."""
        print(
            f"Starting ACO optimization with {self.n_ants} ants "
            f"and {self.n_iterations} iterations"
        )
        print(
            f"Parameters: alpha={self.alpha}, beta={self.beta}, "
            f"rho={self.rho}, q0={self.q0}"
        )
        start_time = time.time()

        for iteration in range(self.n_iterations):
            iteration_solutions = []
            iteration_best_solution = None
            iteration_best_fitness = float("inf")

            for ant_id in range(self.n_ants):
                solution = self.construct_solution(ant_id, iteration)
                iteration_solutions.append(solution)

                if solution.fitness < iteration_best_fitness:
                    iteration_best_fitness = solution.fitness
                    iteration_best_solution = solution

                if solution.fitness < self.best_fitness:
                    self.best_fitness = solution.fitness
                    self.best_solution = solution

            self.update_pheromones(iteration_solutions)
            self.convergence_history.append(
                {
                    "iteration": iteration + 1,
                    "best_fitness": self.best_fitness,
                    "iteration_best_fitness": iteration_best_fitness,
                    "avg_fitness": float(
                        np.mean([solution.fitness for solution in iteration_solutions])
                    ),
                    "worst_fitness": max(
                        solution.fitness for solution in iteration_solutions
                    ),
                }
            )
            self.iteration_best_solutions.append(iteration_best_solution)

            if (iteration + 1) % 20 == 0:
                print(
                    f"Iteration {iteration + 1}: "
                    f"Best fitness = {self.best_fitness:.4f}"
                )

        total_time = time.time() - start_time
        print(f"ACO optimization completed in {total_time:.2f} seconds")
        print(f"Best fitness found: {self.best_fitness:.4f}")
        return self.best_solution

In [ ]:
def create_aco_example() -> Tuple[List[ACOComponent], List[ACOSupplier]]:
    """Create the five-component, four-supplier example instance."""
    components = [
        ACOComponent("Component_1_Critical", 0.95, 460, 0.90),
        ACOComponent("Component_2_Standard", 0.65, 810, 0.85),
        ACOComponent("Component_3_Medium", 0.75, 320, 0.88),
        ACOComponent("Component_4_Low", 0.45, 550, 0.80),
        ACOComponent("Component_5_High", 0.85, 280, 0.92),
    ]

    suppliers = [
        ACOSupplier(
            name="Supplier_A",
            unit_costs={
                "Component_1_Critical": 15,
                "Component_2_Standard": 8,
                "Component_3_Medium": 12,
                "Component_4_Low": 6,
                "Component_5_High": 18,
            },
            reliability_score=0.85,
            quality_scores={
                "Component_1_Critical": 0.87,
                "Component_2_Standard": 0.86,
                "Component_3_Medium": 0.88,
                "Component_4_Low": 0.84,
                "Component_5_High": 0.89,
            },
            capacity_utilization=0.75,
            lead_time=14,
            max_capacity=2000,
        ),
        ACOSupplier(
            name="Supplier_B",
            unit_costs={
                "Component_1_Critical": 18,
                "Component_2_Standard": 7,
                "Component_3_Medium": 14,
                "Component_4_Low": 8,
                "Component_5_High": 20,
            },
            reliability_score=0.92,
            quality_scores={
                "Component_1_Critical": 0.94,
                "Component_2_Standard": 0.91,
                "Component_3_Medium": 0.93,
                "Component_4_Low": 0.90,
                "Component_5_High": 0.95,
            },
            capacity_utilization=0.65,
            lead_time=10,
            max_capacity=2500,
        ),
        ACOSupplier(
            name="Supplier_C",
            unit_costs={
                "Component_1_Critical": 16,
                "Component_2_Standard": 9,
                "Component_3_Medium": 13,
                "Component_4_Low": 7,
                "Component_5_High": 19,
            },
            reliability_score=0.88,
            quality_scores={
                "Component_1_Critical": 0.90,
                "Component_2_Standard": 0.89,
                "Component_3_Medium": 0.91,
                "Component_4_Low": 0.87,
                "Component_5_High": 0.92,
            },
            capacity_utilization=0.80,
            lead_time=12,
            max_capacity=1800,
        ),
        ACOSupplier(
            name="Supplier_D",
            unit_costs={
                "Component_1_Critical": 14,
                "Component_2_Standard": 10,
                "Component_3_Medium": 11,
                "Component_4_Low": 9,
                "Component_5_High": 17,
            },
            reliability_score=0.82,
            quality_scores={
                "Component_1_Critical": 0.85,
                "Component_2_Standard": 0.83,
                "Component_3_Medium": 0.86,
                "Component_4_Low": 0.81,
                "Component_5_High": 0.87,
            },
            capacity_utilization=0.70,
            lead_time=16,
            max_capacity=2200,
        ),
    ]
    return components, suppliers


components, suppliers = create_aco_example()
print(f"Created {len(components)} components and {len(suppliers)} suppliers")
print(f"Components: {[component.name for component in components]}")
print(f"Suppliers: {[supplier.name for supplier in suppliers]}")

In [ ]:
aco = AntColonyOptimizer(
    components=components,
    suppliers=suppliers,
    n_ants=30,
    n_iterations=100,
    alpha=1.0,
    beta=2.0,
    rho=0.1,
    q0=0.9,
    cost_weight=0.6,
    reliability_weight=0.4,
)

best_solution = aco.optimize()

print("\n=== BEST-FOUND ACO SOLUTION ===")
print(f"Fitness score: {best_solution.fitness:.4f}")
print(f"Total annual cost: ${best_solution.total_cost:,.2f}")
print(f"Demand-weighted reliability score: {best_solution.reliability_score:.3f}")
print(f"Demand-weighted quality score: {best_solution.quality_score:.3f}")

print("\n=== SOURCING PLAN ===")
for component_name, allocations in best_solution.allocations.items():
    print(f"\n{component_name}:")
    if len(allocations) == 1:
        supplier_name = next(iter(allocations))
        print(f"  Single sourcing: 100% from {supplier_name}")
    else:
        print("  Dual sourcing:")
        for supplier_name, allocation in allocations.items():
            print(f"    {allocation * 100:.0f}% from {supplier_name}")

recomputed = aco.calculate_solution_fitness(best_solution.allocations)
assert np.isclose(recomputed[0], best_solution.fitness)
assert np.isclose(recomputed[1], best_solution.total_cost)
assert np.isclose(recomputed[2], best_solution.reliability_score)
assert np.isclose(recomputed[3], best_solution.quality_score)
assert all(
    np.isclose(sum(allocation.values()), 1.0)
    for allocation in best_solution.allocations.values()
)

In [ ]:
def analyze_convergence(optimizer: AntColonyOptimizer) -> pd.DataFrame:
    """Summarize best-so-far progress and recent sourcing-mode patterns."""
    convergence_df = pd.DataFrame(optimizer.convergence_history)
    if convergence_df.empty:
        raise ValueError("Run the optimizer before analyzing convergence.")

    print("=== CONVERGENCE ANALYSIS ===")
    improvements = (
        convergence_df["best_fitness"].shift(1) - convergence_df["best_fitness"]
    ).iloc[1:]
    convergence_iteration = None

    for position, improvement in enumerate(improvements.tolist(), start=2):
        previous_best = convergence_df.iloc[position - 2]["best_fitness"]
        if improvement < 0.001 * abs(previous_best):
            convergence_iteration = position
            break

    if convergence_iteration is not None:
        print(
            "First iteration with an improvement below 0.1% of the preceding "
            f"best fitness: {convergence_iteration}"
        )
    else:
        print(
            "No such small-improvement step was detected across "
            f"{len(convergence_df)} recorded iterations."
        )
    print(f"Final best fitness: {convergence_df.iloc[-1]['best_fitness']:.4f}")

    recent_solutions = optimizer.iteration_best_solutions[-10:]
    strategy_signatures = {
        tuple(
            "S" if len(solution.allocations[name]) == 1 else "D"
            for name in sorted(solution.allocations)
        )
        for solution in recent_solutions
    }
    print(
        "Distinct single/dual sourcing patterns among the last "
        f"{len(recent_solutions)} iteration-best plans: {len(strategy_signatures)}"
    )
    print(
        "This count distinguishes only sourcing modes by component; it does not "
        "count different supplier identities or allocation details."
    )
    return convergence_df


convergence_df = analyze_convergence(aco)

In [ ]:
def compare_with_baselines(
    components: List[ACOComponent],
    suppliers: List[ACOSupplier],
    optimizer: AntColonyOptimizer,
    best_solution: ACOSolution,
) -> pd.DataFrame:
    """Compare the ACO plan with cheapest and most-reliable single-source plans."""
    cheapest_allocations = {}
    most_reliable_allocations = {}

    for component in components:
        cheapest_supplier = min(
            suppliers, key=lambda supplier: supplier.unit_costs[component.name]
        )
        most_reliable_supplier = max(
            suppliers, key=lambda supplier: supplier.reliability_score
        )
        cheapest_allocations[component.name] = {cheapest_supplier.name: 1.0}
        most_reliable_allocations[component.name] = {
            most_reliable_supplier.name: 1.0
        }

    cheapest_fitness, cheapest_cost, cheapest_reliability, _ = (
        optimizer.calculate_solution_fitness(cheapest_allocations)
    )
    reliable_fitness, reliable_cost, reliable_reliability, _ = (
        optimizer.calculate_solution_fitness(most_reliable_allocations)
    )

    comparison_data = [
        {
            "Method": "ACO (best found)",
            "Total_Cost": best_solution.total_cost,
            "Reliability": best_solution.reliability_score,
            "Fitness": best_solution.fitness,
            "Dual_Sourcing": sum(
                len(allocations) > 1
                for allocations in best_solution.allocations.values()
            ),
        },
        {
            "Method": "Cheapest (single sourcing)",
            "Total_Cost": cheapest_cost,
            "Reliability": cheapest_reliability,
            "Fitness": cheapest_fitness,
            "Dual_Sourcing": 0,
        },
        {
            "Method": "Most reliable (single sourcing)",
            "Total_Cost": reliable_cost,
            "Reliability": reliable_reliability,
            "Fitness": reliable_fitness,
            "Dual_Sourcing": 0,
        },
    ]
    comparison_df = pd.DataFrame(comparison_data)
    print("=== REFERENCE-PLAN COMPARISON ===")
    print(comparison_df.to_string(index=False))

    if not np.isclose(cheapest_cost, 0.0):
        cost_difference = (
            (cheapest_cost - best_solution.total_cost) / cheapest_cost * 100.0
        )
        print("\nACO best-found plan relative to cheapest single sourcing:")
        print(f"  Cost difference: {cost_difference:+.1f}%")
    if not np.isclose(cheapest_reliability, 0.0):
        reliability_change = (
            (best_solution.reliability_score - cheapest_reliability)
            / cheapest_reliability
            * 100.0
        )
        print(f"  Reliability change: {reliability_change:+.1f}%")
    return comparison_df


comparison_df = compare_with_baselines(
    components, suppliers, aco, best_solution
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Ant Colony Optimization Analysis", fontsize=16, fontweight="bold")

ax1 = axes[0, 0]
ax1.plot(
    convergence_df["iteration"],
    convergence_df["best_fitness"],
    color="blue",
    linewidth=2,
    label="Best-so-far fitness",
)
ax1.plot(
    convergence_df["iteration"],
    convergence_df["iteration_best_fitness"],
    color="green",
    linestyle="--",
    alpha=0.8,
    label="Iteration-best fitness",
)
ax1.plot(
    convergence_df["iteration"],
    convergence_df["avg_fitness"],
    color="red",
    linestyle=":",
    alpha=0.8,
    label="Mean ant fitness",
)
ax1.fill_between(
    convergence_df["iteration"],
    convergence_df["iteration_best_fitness"],
    convergence_df["worst_fitness"],
    alpha=0.15,
    color="gray",
    label="Iteration-best to worst range",
)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Fitness (lower is better; dimensionless score)")
ax1.set_title("Best-So-Far and Iteration Fitness")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
pheromone_matrix = np.array(
    [
        [aco.pheromone[component.name][supplier.name] for supplier in suppliers]
        for component in components
    ]
)
component_labels = [component.name.replace("_", " ") for component in components]
supplier_labels = [supplier.name.replace("Supplier_", "") for supplier in suppliers]
image = ax2.imshow(pheromone_matrix, cmap="YlOrRd", aspect="auto")
ax2.set_xticks(range(len(supplier_labels)))
ax2.set_yticks(range(len(component_labels)))
ax2.set_xticklabels(supplier_labels)
ax2.set_yticklabels(component_labels)
ax2.set_xlabel("Supplier")
ax2.set_ylabel("Component")
ax2.set_title("Final Pheromone Trail Intensity")
for row in range(len(component_labels)):
    for column in range(len(supplier_labels)):
        ax2.text(
            column,
            row,
            f"{pheromone_matrix[row, column]:.2f}",
            ha="center",
            va="center",
            color="black",
        )
fig.colorbar(image, ax=ax2, label="Pheromone intensity (algorithmic units)")

ax3 = axes[1, 0]
positions = np.arange(len(comparison_df))
bar_width = 0.25
cost_bars = ax3.bar(
    positions - bar_width,
    comparison_df["Total_Cost"],
    bar_width,
    label="Annual cost (currency units)",
    color="lightcoral",
    alpha=0.85,
)
reliability_bars = ax3.bar(
    positions,
    comparison_df["Reliability"] * 10000,
    bar_width,
    label="Reliability score × 10,000",
    color="lightblue",
    alpha=0.85,
)
dual_bars = ax3.bar(
    positions + bar_width,
    comparison_df["Dual_Sourcing"] * 1000,
    bar_width,
    label="Dual-sourced components × 1,000",
    color="lightgreen",
    alpha=0.85,
)
ax3.set_xlabel("Reference plan")
ax3.set_ylabel("Displayed value (series use different scales)")
ax3.set_title("Cost, Reliability, and Sourcing-Mode Comparison")
ax3.set_xticks(positions)
ax3.set_xticklabels(comparison_df["Method"], rotation=25, ha="right")
ax3.legend(fontsize=8)
for bars in (cost_bars, reliability_bars, dual_bars):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax3.annotate(
                f"{height:.0f}",
                (bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=7,
            )

ax4 = axes[1, 1]
strategy_counts = {
    "Single sourcing": sum(
        len(allocations) == 1 for allocations in best_solution.allocations.values()
    ),
    "Dual sourcing": sum(
        len(allocations) > 1 for allocations in best_solution.allocations.values()
    ),
}
ax4.pie(
    strategy_counts.values(),
    labels=[f"{name} ({count})" for name, count in strategy_counts.items()],
    autopct="%1.1f%%",
    colors=["lightcoral", "lightblue"],
    startangle=90,
)
ax4.set_title("Sourcing Mode by Component in Best-Found Plan")

plt.tight_layout()
plt.show()

In [ ]:
def analyze_pheromone_patterns(
    components: List[ACOComponent],
    suppliers: List[ACOSupplier],
    optimizer: AntColonyOptimizer,
    best_solution: ACOSolution,
) -> None:
    """Inspect final pheromone levels and the assignments in the best-found plan."""
    print("=== PHEROMONE ANALYSIS ===")

    for component in components:
        print(
            f"\n{component.name} "
            f"(criticality: {component.criticality_score:.2f}):"
        )
        supplier_pheromones = sorted(
            (
                (supplier.name, optimizer.pheromone[component.name][supplier.name])
                for supplier in suppliers
            ),
            key=lambda item: item[1],
            reverse=True,
        )
        print("  Pheromone levels:")
        for rank, (supplier_name, pheromone_level) in enumerate(
            supplier_pheromones, start=1
        ):
            supplier = next(item for item in suppliers if item.name == supplier_name)
            print(f"    {rank}. {supplier_name}: {pheromone_level:.3f}")
            print(
                f"       Unit cost: ${supplier.unit_costs[component.name]}, "
                f"reliability: {supplier.reliability_score:.2f}"
            )

        allocations = best_solution.allocations[component.name]
        if len(allocations) == 1:
            supplier_name = next(iter(allocations))
            print(f"  Best-found assignment: single sourcing from {supplier_name}")
        else:
            print("  Best-found assignment: dual sourcing")
            for supplier_name, allocation in allocations.items():
                print(f"    {allocation * 100:.0f}% from {supplier_name}")

    all_pheromones = [
        optimizer.pheromone[component.name][supplier.name]
        for component in components
        for supplier in suppliers
    ]
    print("\n=== PHEROMONE DISTRIBUTION ===")
    print(f"Mean trail intensity: {np.mean(all_pheromones):.3f}")
    print(f"Standard deviation: {np.std(all_pheromones):.3f}")
    print(f"Minimum trail intensity: {np.min(all_pheromones):.3f}")
    print(f"Maximum trail intensity: {np.max(all_pheromones):.3f}")
    if np.min(all_pheromones) > 0:
        print(f"Maximum-to-minimum intensity ratio: {np.max(all_pheromones) / np.min(all_pheromones):.1f}x")


analyze_pheromone_patterns(components, suppliers, aco, best_solution)

## Interpreting the method

### What ACO contributes

Pheromone-based search combines information from many constructed plans. The heuristic gives an initial preference using the modeled supplier attributes, while pheromone makes subsequent choices responsive to the plans found by the colony. The best-so-far record supports progress tracking; it does not make the search deterministic or guarantee a global optimum.

### Reading the outputs

- A non-increasing best-so-far fitness indicates that the recorded incumbent has not been replaced by a worse plan. It does not by itself prove that the search has converged to a global optimum.
- The convergence analysis reports the first step whose incumbent improvement is below the stated relative threshold. A single small improvement is a descriptive signal, not a statistical or mathematical convergence guarantee.
- The pheromone heatmap shows the accumulated trail values after the final update. Trail intensity reflects the algorithm's history and reinforcement rule; it is not a probability, supplier quality score, or proof that an assignment is optimal.
- The reference plans are deliberately simple single-sourcing plans: one selects the least-cost supplier for each component, and the other selects the supplier with the highest reliability score. They are comparison points, not additional optimization algorithms.
- The best-found plan's single/dual count describes its sourcing mode by component. The construction rule limits dual sourcing to qualifying high-criticality components, and each dual allocation uses a 70%/30% split.

### Strengths, limitations, and appropriate use

ACO is useful when a problem can be represented by constructive decisions and a good feasible plan is valuable even when a global-optimality certificate is unavailable. Pheromone learning, probabilistic exploration, and population-based construction are central features of this method. Their effectiveness depends on the construction rules, objective scaling, data, and parameter choices.

This example is intentionally limited: the dual-sourcing decision is generated by a fixed probability rule rather than selected by direct comparison of single and dual allocations; supplier capacity limits and quality requirements are not enforced; and the fixed 70%/30% split may not suit every application. Consequently, its numerical results should be interpreted as outcomes for the stated model, not as operational recommendations without further validation.

For larger or more realistic instances, natural extensions include representing additional operational constraints in the same solution-construction and fitness framework, and testing parameter choices on the intended data. Such extensions require explicit modeling and validation; they are not implemented in this example.